# <font color="blue"><b>Vision-Language Models: From Zero-Shot Applications to Fine-Tuning and Multimodal RAG</b></font>

---

## Problem Statement

Vision-Language Models (VLMs) let a single model read an image and text together — answering
questions about a photo, transcribing and reasoning over a scanned document, or retrieving the
right image for a query. In this assignment you will work with **Qwen2.5-VL-3B-Instruct** across
four progressively harder settings:

1. **Zero-shot applications** — VQA, captioning, chain-of-thought counting, and document
   intelligence (OCR-VQA + document understanding), all built on the same
   `messages → apply_chat_template → generate` pattern used in Lecture 2.
2. **Parameter-efficient fine-tuning** — attach 4-bit + LoRA adapters (as in Lecture 1) and
   fine-tune on the VQA dataset, then compare answers **before vs after** fine-tuning.
3. **Multimodal RAG** — build a small mixed-domain image corpus (CIFAR-10 + DocVQA +
   InfographicVQA), extract OCR + captions, index it two ways (FAISS text embeddings and ColPali
   visual late-interaction), fuse the two rankings with **Reciprocal Rank Fusion (RRF)**, and
   generate a grounded answer — exactly the Lecture 3 pipeline.

The notebook is organised into twelve numbered steps. Steps 1–3 (imports & seed, dataset +
model loading, GPU helpers) are **provided** and carry no marks; the graded steps sum to **100**.

## Resources and Runtime

- Run this notebook in **Google Colab** with a **T4 GPU** (`Runtime → Change runtime type → GPU`).
- The notebook installs all Python packages itself (`transformers`, `qwen-vl-utils`, `peft`,
  `trl`, `bitsandbytes`, `easyocr`, `colpali-engine`, `faiss-cpu`, `sentence-transformers`).
- All datasets and models are public and downloaded by code — no API key, login, or Drive mount
  is required.
- A small image subset and short training run are used deliberately so everything finishes in
  one Colab session.

## What You Must Submit

Submit one completed `.ipynb`. Every `## CODE HERE ##` placeholder and every **[Your Answer]**
reflection must be completed, and all outputs (generations, plots, tables, retrieved images)
must remain visible. The notebook must run top to bottom after a runtime restart.

## Assessment Principles

- The prompt for every generation task must use `processor.apply_chat_template(...,
  add_generation_prompt=True)` — never a hand-built special-token string.
- LoRA fine-tuning must use a 4-bit NF4 backbone (`BitsAndBytesConfig`) with a `bf16` compute
  dtype, matching the Lecture 1 hands-on recipe, so it fits a free T4.
- The **same** three sample questions must be used for the before/after fine-tuning comparison
  in Step 8, and the **same** evaluation queries must be used across FAISS, ColPali, and RRF in
  Step 11, so the comparisons are fair.
- Written answers must cite your own measured numbers, generations, and retrieved results —
  not generic descriptions.
- Equivalent correct implementations are accepted; copying hard-coded outputs is not.


### <font color="blue"><b>Step 1: Imports, Installs and Seed</b></font>  *(provided — just run)*

Install the packages this notebook needs and import everything used throughout. A single seed
is fixed across `random`, NumPy and PyTorch so sampling and shuffling are reproducible.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 1: Installs, Imports and Seed
# ---PROVIDED ------------------------------------------------
# ============================================================
!pip install -q datasets sentence-transformers faiss-cpu easyocr
!pip install -q -U "colpali-engine>=0.3.1"
!pip install -q -U git+https://github.com/huggingface/trl.git bitsandbytes peft qwen-vl-utils

# Colab ships an old torchao that peft's import check rejects (ImportError: incompatible
# torchao version). Nothing here needs torchao, so just remove it rather than upgrading it.
!pip uninstall -y -q torchao

import os, gc, math, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from PIL import Image
from datasets import load_dataset
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoTokenizer,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


### <font color="blue"><b>Step 2: Dataset and Model Loading</b></font>  *(provided — just run)*

Load the three datasets used in the zero-shot / fine-tuning half of the notebook
(`miniVQAv2` for VQA + LoRA, `Quantity-Reasoning-VQA-23K` for chain-of-thought counting,
`DocVQA` for document intelligence), and load `Qwen2.5-VL-3B-Instruct` with its processor.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 2: Dataset and Model Loading
# ---PROVIDED ------------------------------------------------
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

vqa_dataset  = load_dataset("dutta18/miniVQAv2DatasetForCollab", split="train")
qty_dataset  = load_dataset("dutta18/Quantity-Reasoning-VQA-23K", split="val[:100]")
docvqa_data  = load_dataset("lmms-lab-encoder/DocVQA", "DocVQA", split="test[:100]")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",
)

min_pixels = 256 * 28 * 28
max_pixels = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=min_pixels, max_pixels=max_pixels, use_fast=True
)

print("Model + processor ready.")
print("VQA:", vqa_dataset, "\nQuantity-Reasoning:", qty_dataset, "\nDocVQA:", docvqa_data)


### <font color="blue"><b>Step 3: Helper Functions</b></font>  *(provided — just run)*

Small utilities reused across the notebook: `free_memory` (VRAM cleanup between model loads)
and `count_trainable` (trainable-vs-total parameter counts, used after attaching LoRA).

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 3: Helper Functions
# ---PROVIDED ------------------------------------------------
# ============================================================
def free_memory(*names):
    # Free GPU + RAM. Pass names of globals to delete first: free_memory("model").
    # empty_cache() can only release memory with no live references left, so anything
    # still held by a variable, by the last exception's traceback, or by Jupyter's
    # output history must be dropped first.
    import sys

    # Delete the named globals in the CALLER's namespace (the notebook).
    caller_globals = sys._getframe(1).f_globals
    for name in names:
        caller_globals.pop(name, None)

    # An OOM (or any) exception keeps its traceback alive, and that traceback keeps
    # every local variable in every frame alive -- including big tensors.
    sys.last_traceback = None
    sys.last_value = None
    sys.last_type = None

    # Jupyter/Colab pins the result of every past cell in Out[] / _oh and in _, __, ___.
    # Clear those exact containers by hand -- do NOT use %reset_selective here, its
    # argument is a REGEX and a pattern like "_" silently deletes every global whose
    # name contains an underscore (MODEL_ID, vqa_dataset, ...).
    try:
        ip = get_ipython()
        for store in ("Out", "_oh"):
            obj = ip.user_ns.get(store)
            if isinstance(obj, dict):
                obj.clear()
        for hist in ("_", "__", "___", "_i", "_ii", "_iii"):
            if hist in ip.user_ns:
                ip.user_ns[hist] = None
    except NameError:
        pass  # not running under IPython

    gc.collect()
    gc.collect()  # second pass catches objects freed by the first (reference cycles)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated | "
              f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

def count_trainable(m):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    return trainable, total, 100 * trainable / total

print("Helpers ready.")


## <font color="blue"><b>Step 4: Zero-Shot Visual Question Answering</b></font>  <font color="red">**[9 marks]**</font>


Build the standard VLM inference pipeline: a `messages` list with an image + text
turn, rendered through `processor.apply_chat_template(..., add_generation_prompt=True)`, fed to
`processor(...)`, and decoded after `model.generate(...)`. Wrap this in a reusable
`vqa_generate(image, question)` function and run it on **3 samples** from `vqa_dataset`.

In [ ]:
# ============================================================
#  Step 4.1: Zero-Shot VQA
# ============================================================
def vqa_generate(image, question, max_new_tokens=128):
    messages = [
        {
            "role": "user",
            "content": [
                ## CODE HERE ## (image content dict)
                ## CODE HERE ## (text content dict, using `question`)
            ],
        }
    ]

    ## CODE HERE ##
    # 1. text = processor.apply_chat_template(...)
    # 2. image_inputs, video_inputs = process_vision_info(messages)
    # 3. inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
    #                        padding=True, return_tensors="pt").to(model.device)
    # 4. generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # 5. trim the prompt tokens off `generated_ids` before decoding

    return output_text


# ---- Run on 3 samples ----
# Cache the base-model answers now, while the base model is loaded. Step 8 compares
# against these instead of reloading the 7 GB base model next to the fine-tuned one.
SAMPLE_IDS = [1, 10, 25]
BASE_ANSWERS = {}

for idx in SAMPLE_IDS:
    image = vqa_dataset["image"][idx]
    question = vqa_dataset["question"][idx]
    BASE_ANSWERS[idx] = vqa_generate(image, question)
    print(f"[{idx}] Q: {question}")
    print("A:", BASE_ANSWERS[idx])
    print("-" * 60)


### ✍️ Step 4 Reflection  <font color="red">**[3 marks]**</font>
**[Your Answer]** must cover:
- What each element of the `messages` list represents, and why `apply_chat_template` is used instead of hand-writing special tokens. <font color="red">**[1]**</font>
- The three generations you printed — do they look correct given the images and questions? <font color="red">**[1]**</font>
- Why the prompt tokens must be trimmed off `generated_ids` before decoding. <font color="red">**[1]**</font>


## <font color="blue"><b>Step 5: Captioning and Chain-of-Thought Reasoning</b></font>  <font color="red">**[14 marks]**</font>


### <font color="blue"><b>Step 5.1: Image Captioning</b></font>  <font color="red">**[5 marks]**</font>


Write a captioning **system prompt** that asks for a descriptive, visually-grounded
caption (no hallucinated detail), then extend `vqa_generate`-style inference to accept a system
prompt. Run it on **3 images** from `vqa_dataset`.

In [ ]:
# ============================================================
#  Step 5.1: Captioning
# ============================================================
captioning_system_prompt = """
## CODE HERE ##  (a captioning instruction: descriptive, visually-grounded, no hallucinated detail)
"""

def generate_with_system(system_prompt, image, question=None, max_new_tokens=256):
    content = [{"type": "image", "image": image}]
    ## CODE HERE ##  (append a text content dict to `content` ONLY if `question` is given)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": content},
    ]
    ## CODE HERE ##  (same 5-step generate/decode pipeline as Step 4)

    return output_text


for idx in [5, 40, 81]:
    image = vqa_dataset["image"][idx]
    print(f"[{idx}] Caption:", generate_with_system(captioning_system_prompt, image))
    print("-" * 60)


### <font color="blue"><b>Step 5.2: Chain-of-Thought Counting</b></font>  <font color="red">**[5 marks]**</font>


Using `qty_dataset` (object-counting questions), write a **direct-answer** system
prompt and a **chain-of-thought** system prompt that reasons region-by-region before giving a
final count. Run both on the **same 2 images** and compare.

In [ ]:
# ============================================================
#  Step 5.2: Chain-of-Thought Counting
# ============================================================
direct_system_prompt = "## CODE HERE ##"  # answer with only the final number, no explanation

cot_system_prompt = """
## CODE HERE ##
# A counting assistant that reasons step by step:
# breaks the image into regions, counts within each, sums them,
# and states "Hence, the final answer is N."
"""

for idx in [17, 35]:
    question = qty_dataset["question"][idx]
    image = qty_dataset["image"][idx]

    print(f"[{idx}] Q: {question}")
    print("Direct:", generate_with_system(direct_system_prompt, image, question))
    print("CoT:", generate_with_system(cot_system_prompt, image, question, max_new_tokens=400))
    print("=" * 60)


### ✍️ Step 5 Reflection  <font color="red">**[4 marks]**</font>
**[Your Answer]** must cover:
- What makes a captioning system prompt "visually grounded" and how it reduces hallucination. <font color="red">**[1]**</font>
- Cite the direct-vs-CoT outputs for both images — did CoT reasoning change the final count, and was it more accurate? <font color="red">**[2]**</font>
- One case (from your own runs) where CoT still got the count wrong, and why long reasoning does not guarantee correctness. <font color="red">**[1]**</font>


## <font color="blue"><b>Step 6: Document Intelligence</b></font>  <font color="red">**[10 marks]**</font>


Reuse `generate_with_system` for two document tasks on `docvqa_data`: (a)
**OCR-VQA** — answer a specific question about a document image, and (b) **document
understanding** — produce a structured summary of everything visible in a document (headings,
tables, key numbers) without being asked a specific question.

In [ ]:
# ============================================================
#  Step 6: Document Intelligence
# ============================================================
doc_qa_system_prompt = """
## CODE HERE ##
# An OCR + document-QA assistant: read all visible text, understand layout/structure,
# then answer the user's question using only information present in the document.
"""

doc_summary_system_prompt = """
## CODE HERE ##
# A document-understanding assistant: OCR the full document, then present ALL visible
# information (headings, paragraphs, tables, labels, numbers) in an organised summary,
# without answering any specific question.
"""

# ---- OCR-VQA: answer a specific question ----
for idx in [30, 90]:
    question = docvqa_data["question"][idx]
    image = docvqa_data["image"][idx]
    print(f"[{idx}] Q: {question}")
    print("A:", generate_with_system(doc_qa_system_prompt, image, question))
    print("-" * 60)

# ---- Document Understanding: full summary, no question ----
for idx in [50]:
    image = docvqa_data["image"][idx]
    print(f"[{idx}] Summary:", generate_with_system(doc_summary_system_prompt, image, max_new_tokens=512))


### ✍️ Step 6 Reflection  <font color="red">**[4 marks]**</font>
**[Your Answer]** must cover:
- How the OCR-VQA prompt differs from the document-understanding prompt, and why the latter must not "answer a question" that wasn't asked. <font color="red">**[2]**</font>
- Whether the OCR-VQA answers matched the ground truth in `docvqa_data['answers']` for your two samples — quote them. <font color="red">**[1]**</font>
- One failure you observed (misread number, missed table, wrong field) and a likely cause. <font color="red">**[1]**</font>


## <font color="blue"><b>Step 7: LoRA Fine-Tuning Setup and Training</b></font>  <font color="red">**[22 marks]**</font>


## Free the Base VLM Before Fine-Tuning


In [ ]:
# The base `model` is not needed during fine-tuning. Free it so the 4-bit `ft_model`,
# LoRA adapters, optimizer state and training activations all fit on a T4.
# Step 8 compares against BASE_ANSWERS (cached in Step 4), so the base model is not
# reloaded until Step 9.
free_memory("model")


### <font color="blue"><b>Step 7.1: 4-bit Backbone, LoRA Config, Collator and Trainer</b></font>  <font color="red">**[10 marks]**</font>


Set up parameter-efficient fine-tuning exactly as in the Lecture 1 hands-on:
load a **4-bit NF4** backbone (`BitsAndBytesConfig`), attach a `LoraConfig` targeting the
attention + MLP projections, write a `custom_data_collator` that turns `(image, question)`
pairs into a model-ready batch with `labels`, and wrap it all in an `SFTTrainer` /
`SFTConfig`.

In [ ]:
# ============================================================
#  Step 7.1: 4-bit Backbone + LoRA Config + Collator + Trainer
# ============================================================
bnb_config = BitsAndBytesConfig(
    ## CODE HERE ##  (load_in_4bit, double_quant, quant_type="nf4", compute_dtype=bfloat16)
)

ft_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
    quantization_config=bnb_config,
)

peft_config = LoraConfig(
    ## CODE HERE ##
    # lora_alpha, lora_dropout, r, bias="none",
    # target_modules covering q_proj/k_proj/v_proj/o_proj (+ MLP projections if you like),
    # task_type="CAUSAL_LM"
)

def custom_data_collator(samples):
    batch_messages = []
    for sample in samples:
        ## CODE HERE ##
        # Build a `messages` list: one user turn with an image content dict
        # and a text content dict from sample["question"].
        batch_messages.append(messages)

    batch_text = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]
    batch_images = [m[0]["content"][0]["image"] for m in batch_messages]

    model_inputs = processor(text=batch_text, images=batch_images, padding=True, return_tensors="pt")
    ## CODE HERE ##  (model_inputs["labels"] = a clone of input_ids)
    return model_inputs


training_args = SFTConfig(
    output_dir="qwen_vqa_lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    remove_unused_columns=False,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=vqa_dataset,
    peft_config=peft_config,
    data_collator=custom_data_collator,
)

trainable, total, pct = count_trainable(trainer.model)
print(f"Trainable: {trainable:,} / {total:,} ({pct:.3f}%)")


### <font color="blue"><b>Step 7.2: Run Training and Plot Loss</b></font>  <font color="red">**[6 marks]**</font>


Train, then plot the per-logging-step loss from `trainer.state.log_history`.

In [ ]:
# ============================================================
#  Step 7.2: Train + Loss Curve
# ============================================================
## CODE HERE ##  (trainer.train())

## CODE HERE ##
# Pull "loss" entries out of trainer.state.log_history and plot loss vs step.


### ✍️ Step 7 Reflection  <font color="red">**[6 marks]**</font>
**[Your Answer]** must cover:
- Each `BitsAndBytesConfig` setting in plain language, and why the compute dtype is `bfloat16` here (T4/L4 hands-on convention). <font color="red">**[2]**</font>
- Your `LoraConfig` choices (`r`, `lora_alpha`, `target_modules`) and the measured trainable-parameter percentage. <font color="red">**[2]**</font>
- What the collator's `labels = input_ids.clone()` line means for what the model is trained to predict, and why `remove_unused_columns=False` is required here. <font color="red">**[1]**</font>
- What the loss curve shows — did it fall smoothly, and does it look like more epochs would keep helping? <font color="red">**[1]**</font>


## <font color="blue"><b>Step 8: Before vs After Fine-Tuning</b></font>  <font color="red">**[9 marks]**</font>


Using the **same 3 VQA samples from Step 4**, compare the base model's answers
(already printed in Step 4) against the fine-tuned model's answers. Build a small
`vqa_generate_ft` using the trained `trainer.model` (a `PeftModel`) and print the answers
side by side.

In [ ]:
# ============================================================
#  Step 8.1: Before vs After Comparison
# ============================================================
def vqa_generate_ft(image, question, max_new_tokens=128):
    ## CODE HERE ##
    # Same pipeline as `vqa_generate`, but call trainer.model.generate(...) instead of
    # the original `model`.
    return output_text


# Base answers come from the Step 4 cache (BASE_ANSWERS) so the base model does not
# have to sit in GPU memory next to the fine-tuned one.
for idx in SAMPLE_IDS:
    image = vqa_dataset["image"][idx]
    question = vqa_dataset["question"][idx]
    print(f"[{idx}] Q: {question}")
    print("  Base      :", BASE_ANSWERS[idx])
    print("  Fine-tuned:", vqa_generate_ft(image, question))
    print("-" * 60)


### ✍️ Step 8 Reflection  <font color="red">**[4 marks]**</font>
**[Your Answer]** must cover:
- Cite the base-vs-fine-tuned answers for all 3 samples — did fine-tuning change style, correctness, or both? <font color="red">**[2]**</font>
- Whether one epoch on this small subset was enough to see a clear change, and what you'd try if not (more epochs, higher rank, larger subset). <font color="red">**[2]**</font>


## Free the Fine-Tuning Artifacts and Reload the Base VLM


In [ ]:
# Fine-tuning is done. Drop the optimizer state first (it is often as large as the
# adapters themselves), then free the trainer and the 4-bit model.
if "trainer" in globals() and getattr(trainer, "optimizer", None) is not None:
    trainer.optimizer.zero_grad(set_to_none=True)
    trainer.optimizer = None
free_memory("trainer", "ft_model")

# Step 9 captions every RAG image with the base VLM, so load it back now.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="auto",
)


## <font color="blue"><b>Step 9: Multimodal Dataset and Extraction</b></font>  <font color="red">**[8 marks]**</font>


Build a small mixed-domain image corpus for the RAG pipeline: **20 CIFAR-10**,
**20 DocVQA**, and **20 InfographicVQA** images (as in Lecture 3), each stored as a dict with
`id`, `source`, `label`, `image`. Then extract, per image, an **EasyOCR** text pass and a
caption (reuse `generate_with_system` with a short captioning prompt in place of Florence-2),
and combine both into a `combined_text` field.

In [ ]:
# ============================================================
#  Step 9.1: Unified Dataset + OCR + Captions
# ============================================================
CIFAR_LABELS = {0:"airplane",1:"automobile",2:"bird",3:"cat",4:"deer",
                5:"dog",6:"frog",7:"horse",8:"ship",9:"truck"}

cifar = load_dataset("uoft-cs/cifar10", split="train[:20]")
infography = load_dataset("lmms-lab-encoder/DocVQA", "InfographicVQA", split="validation[:20]")
docvqa_small = load_dataset("lmms-lab-encoder/DocVQA", "DocVQA", split="validation[:20]")

RAG_DATASET = []
## CODE HERE ##
# Append 20 CIFAR-10 items: {"id", "source": "cifar10", "label": CIFAR_LABELS[...],
#                            "image": item["img"].convert("RGB")}
# Append 20 DocVQA items:   {"id", "source": "docvqa", "label": "document",
#                            "image": item["image"].convert("RGB"), "question": item["question"]}
# Append 20 InfographicVQA items similarly with source "infographic"

print(f"Built RAG_DATASET with {len(RAG_DATASET)} images")

# ---- EasyOCR ----
import easyocr
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)

def run_ocr(image):
    results = ocr_reader.readtext(np.array(image), detail=1, paragraph=False)
    ## CODE HERE ##  (keep texts with confidence > 0.4, join with a space)
    return text

for item in RAG_DATASET:
    item["ocr_text"] = run_ocr(item["image"])

# ---- Captioning (reuse Step 5's system prompt) ----
for item in RAG_DATASET:
    ## CODE HERE ##
    # item["caption"] = generate_with_system(captioning_system_prompt, item["image"], max_new_tokens=60)
    pass

# ---- Combine ----
for item in RAG_DATASET:
    ## CODE HERE ##  (item["combined_text"] = caption + " " + ocr_text, stripped)
    pass

print(RAG_DATASET[0]["combined_text"])


### ✍️ Step 9 Reflection  <font color="red">**[2 marks]**</font>
**[Your Answer]** must cover:
- Why the pipeline extracts BOTH a caption and OCR text, instead of just one, giving an example image where OCR alone returns nothing. <font color="red">**[1]**</font>
- One risk of using a VLM caption to build a retrieval index (e.g. missed detail, hallucinated content). <font color="red">**[1]**</font>


## Free the Base VLM Before Loading ColPali


In [ ]:
# Captioning is done. `model` is not needed again until Step 11.2, so free it to make
# room for ColPali's patch embeddings alongside the FAISS text index.
free_memory("model")


## <font color="blue"><b>Step 10: Dual Retrieval: FAISS Text Index and ColPali Visual Index</b></font>  <font color="red">**[12 marks]**</font>


### <font color="blue"><b>Step 10.1: Sentence Embeddings + FAISS</b></font>  <font color="red">**[4 marks]**</font>


Embed every image's `combined_text` with `all-mpnet-base-v2`, build a FAISS
`IndexFlatIP` (cosine similarity via normalised inner product), and write a `text_retrieve`
function. Sanity-check it with a text query.

In [ ]:
# ============================================================
#  Step 10.1: FAISS Text Retrieval
# ============================================================
import faiss
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)

combined_texts = [item["combined_text"] for item in RAG_DATASET]
## CODE HERE ##
# text_embeddings = st_model.encode(..., normalize_embeddings=True, convert_to_numpy=True).astype("float32")

dim = text_embeddings.shape[1]
## CODE HERE ##  (build faiss.IndexFlatIP(dim) and add text_embeddings)

def text_retrieve(query, top_k=5):
    ## CODE HERE ##
    # encode the query the same way, search the index, return list of (idx, score)
    return list(zip(indices[0].tolist(), scores[0].tolist()))

print(text_retrieve("a fluffy pet animal", top_k=5))


### <font color="blue"><b>Step 10.2: ColPali Visual Index</b></font>  <font color="red">**[4 marks]**</font>


Load `vidore/colpali-v1.3`, embed every image as a set of patch vectors, and write
`colpali_retrieve` using **MaxSim** late interaction. Visualise the MaxSim relevance as a
heatmap over one retrieved image's patches.

In [ ]:
# ============================================================
#  Step 10.2: ColPali Visual Retrieval
# ============================================================
from colpali_engine.models import ColPali, ColPaliProcessor

colpali_model = ColPali.from_pretrained("vidore/colpali-v1.3", dtype=torch.bfloat16, device_map="cuda:0")
colpali_proc = ColPaliProcessor.from_pretrained("vidore/colpali-v1.3")
colpali_model.eval()

def embed_images_colpali(images, batch_size=2):
    all_embs = []
    ## CODE HERE ##
    # For each batch: colpali_proc.process_images(batch) -> move tensors to GPU ->
    # torch.inference_mode(): colpali_model(**inputs) -> move to CPU, append per-image tensors
    return all_embs

colpali_image_embeddings = embed_images_colpali([item["image"] for item in RAG_DATASET])

def colpali_retrieve(query, top_k=5):
    ## CODE HERE ##
    # 1. colpali_proc.process_queries([query]) -> GPU -> colpali_model(**inputs)[0] -> CPU
    # 2. MaxSim(query_emb, image_emb) for every image: (query_emb @ image_emb.T).max(dim=1).values.sum()
    # 3. sort descending, return top_k (idx, score) pairs
    return results

print(colpali_retrieve("a fluffy pet animal", top_k=5))


### ✍️ Step 10 Reflection  <font color="red">**[4 marks]**</font>
**[Your Answer]** must cover:
- Why cosine similarity is achieved here by inner product on normalised vectors, and what `IndexFlatIP` computes. <font color="red">**[1]**</font>
- What MaxSim does conceptually (per query-token best-matching patch, summed) and why it can catch detail that captions/OCR miss. <font color="red">**[2]**</font>
- Cite one query where FAISS and ColPali's top-1 results DIFFER — which retriever got it right and why. <font color="red">**[1]**</font>


## <font color="blue"><b>Step 11: RRF Fusion and Grounded Generation</b></font>  <font color="red">**[12 marks]**</font>


### <font color="blue"><b>Step 11.1: Reciprocal Rank Fusion</b></font>  <font color="red">**[4 marks]**</font>


Implement `rrf_retrieve` combining the FAISS and ColPali rankings with **Reciprocal
Rank Fusion**: RRF(d) = Σ 1/(k + rank_r(d)), k=60. Compare FAISS-only, ColPali-only, and RRF
top-3 for the same query.

In [ ]:
# ============================================================
#  Step 11.1: Reciprocal Rank Fusion
# ============================================================
def rrf_retrieve(query, top_k=5, k=60):
    n = len(RAG_DATASET)
    text_results = text_retrieve(query, top_k=n)
    colpali_results = colpali_retrieve(query, top_k=n)

    ## CODE HERE ##
    # 1. Build {idx: rank} dicts (rank starts at 1) for both result lists.
    # 2. rrf_scores[idx] = 1/(k + text_rank.get(idx, n+1)) + 1/(k + colpali_rank.get(idx, n+1))
    #    for every idx in range(n)
    # 3. sort descending by rrf score, return top_k (idx, score) pairs
    return fused[:top_k]

test_query = "a fluffy pet animal"
print("FAISS  :", text_retrieve(test_query, top_k=3))
print("ColPali:", colpali_retrieve(test_query, top_k=3))
print("RRF    :", rrf_retrieve(test_query, top_k=3))


### <font color="blue"><b>Step 11.2: Grounded Generation and Retrieval Accuracy</b></font>  <font color="red">**[4 marks]**</font>


Build 10 evaluation queries (one per CIFAR-10 label present + a few real DocVQA
questions with a known `target_idx`). For each query, retrieve top-3 with FAISS, ColPali and
RRF, and compute **hit@3** (was `target_idx` in the top-3?) for each retriever. Then generate a
grounded answer for one query using `vqa_generate` on the top-1 RRF image, constrained to
"answer only from the image, say 'Not visible' if absent.

## Reload the VLM for Grounded Generation


In [ ]:
# Grounded generation needs the base VLM back alongside the already-loaded ColPali index.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="auto",
)


In [ ]:
# ============================================================
#  Step 11.2: Grounded Generation + Retrieval Accuracy
# ============================================================
EVAL_QUERIES = []
## CODE HERE ##
# Build EVAL_QUERIES: list of {"query": ..., "target_idx": idx} using
#   - one query per distinct CIFAR-10 label ("a photo of a {label}")
#   - a few DocVQA/infographic items' own "question" field, target_idx = that item's id

def hit_at_k(retrieve_fn, queries, k=3):
    ## CODE HERE ##
    # For each query, run retrieve_fn(q["query"], top_k=k), check if q["target_idx"]
    # is among the returned indices; return the fraction of hits.
    return hits / len(queries)

# Cache the scores -- Step 12 reuses them instead of re-running every retrieval.
HIT_RESULTS = {
    "FAISS":   hit_at_k(text_retrieve, EVAL_QUERIES),
    "ColPali": hit_at_k(colpali_retrieve, EVAL_QUERIES),
    "RRF":     hit_at_k(rrf_retrieve, EVAL_QUERIES),
}
for _name, _score in HIT_RESULTS.items():
    print(f"{_name:8s} hit@3: {_score:.2f}")

# ---- Grounded generation on one query ----
grounded_prompt = "Answer ONLY using what is visible in the provided image. Say 'Not visible' if the answer is not present."
demo_query = EVAL_QUERIES[0]["query"]
top1_idx, _ = rrf_retrieve(demo_query, top_k=1)[0]
print("Query:", demo_query)
print("Answer:", generate_with_system(grounded_prompt, RAG_DATASET[top1_idx]["image"], demo_query))


### ✍️ Step 11 Reflection  <font color="red">**[4 marks]**</font>
**[Your Answer]** must cover:
- Explain the RRF formula in your own words and why it needs no score normalisation across retrievers. <font color="red">**[1]**</font>
- Cite the three hit@3 numbers — which retriever performed best overall, and did RRF beat BOTH individual retrievers or just average them? <font color="red">**[2]**</font>
- Why the grounded system prompt reduces hallucination risk compared to an unconstrained "answer the question" prompt. <font color="red">**[1]**</font>


## <font color="blue"><b>Step 12: Final Synthesis</b></font>  <font color="red">**[4 marks]**</font>


Bring the whole notebook together in one comparison. Build a small table/chart
covering: (a) zero-shot vs fine-tuned quality signal from Step 8, and (b) hit@3 for FAISS,
ColPali and RRF from Step 11.

In [ ]:
# ============================================================
#  Step 12: Final Synthesis
# ============================================================
retrieval_summary = pd.DataFrame({
    "retriever": ["FAISS", "ColPali", "RRF"],
    "hit_at_3": [
        ## CODE HERE ##  (reuse the cached HIT_RESULTS values from Step 11.2)
    ],
})
print(retrieval_summary)

## CODE HERE ##
# Bar chart of hit_at_3 by retriever.


### ✍️ Step 12 Reflection  <font color="red">**[2 marks]**</font>
**[Your Answer]** must cover:
- Which method gave the best BALANCE across the notebook (zero-shot quality, fine-tuning gain, retrieval accuracy) on a free Colab GPU, citing your numbers. <font color="red">**[1]**</font>
- One thing you would change with more time/compute (bigger fine-tuning subset, more epochs, higher LoRA rank, larger RAG corpus). <font color="red">**[1]**</font>
